# py-dyneval — R⇄Py parity

8 metrics validated against R `dyneval::calculate_metrics`.

In [1]:
import subprocess, sys, os, json
from pathlib import Path
import pandas as pd, numpy as np
PORT = Path('..').resolve()
DATA = PORT/'data'; DATA.mkdir(exist_ok=True)
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = '/share/software/user/open/netcdf/4.8.1/lib:'+env.get('LD_LIBRARY_PATH','')
subprocess.run(['conda','run','-p','/scratch/users/steorra/env/CMAP','Rscript',
                str(PORT/'tests/r_reference_driver.R'), str(DATA)], check=True, env=env, capture_output=True)
subprocess.run([sys.executable, str(PORT/'tests/_run_candidate.py'), str(DATA)], check=True, capture_output=True)
print('R + Py done')

R + Py done


In [2]:
r = json.loads((DATA/'reference_metrics.json').read_text())
p = json.loads((DATA/'candidate_metrics.json').read_text())
keys = ['correlation','him','edge_flip','isomorphic','F1_branches','F1_milestones','featureimp_cor','featureimp_wcor']
df = pd.DataFrame({'R':[r[k] for k in keys], 'Py':[p[k] for k in keys]}, index=keys)
df['|Δ|'] = (df.R-df.Py).abs()
print(df.round(4).to_string())

                      R      Py     |Δ|
correlation      0.9812  0.9813  0.0001
him              1.0000  1.0000  0.0000
edge_flip        1.0000  1.0000  0.0000
isomorphic       1.0000  1.0000  0.0000
F1_branches      1.0000  1.0000  0.0000
F1_milestones    0.6667  0.6667  0.0000
featureimp_cor   0.9622  0.9005  0.0617
featureimp_wcor  0.9664  0.9548  0.0115
